# PN1C parameter-matched sieve-rung compression competition

## TL;DR

The frozen claim was **not supported**. On the held-out prime-19 to prime-23 transition, the 35-slot fixed 6×6 ARA grid scored **0.470280 bits** of Jensen–Shannon divergence. The 31-slot gap-IID rival won at **0.230999 bits** and won separately in both child halves. Every exact check and the independent reconstruction passed.

This falsifies the narrow claim that this fixed ARA grid and uniform decoder is the best ≤36-slot compressor. It does not falsify the ARA coordinate or PN1's result that local order survives sieve rungs.


## 1. Setup and frozen test

For adjacent circular gaps, the bounded coordinate is

\[x_i=\frac{2g_{i+1}}{g_i+g_{i+1}}\in(0,2),\qquad Z_i=(x_i,x_{i+1}).\]

The target is the prime-23 wheel's 24×24 distribution of \(Z_i\). The parent is the complete prime-19 wheel. Six frozen competitors receive no more than 36 declared scalar slots. The primary prediction requires ARA to have strictly smallest JSD and beat the best rival by at least 1% relative.

Frozen protocol SHA-256: `7DAA061BA790B12461ED60136FD9C50F3A36C10BED472819CFCC08B4B3462DBF`.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from pn1c_compression_test import run_analysis

HERE = Path.cwd()
results = run_analysis(HERE)
print("\nReproduced frozen summary:")
print(json.dumps(results["summary"], indent=2))


  primary stream: completed lift 1/23; survivors=1,586,754
  primary stream: completed lift 6/23; survivors=9,520,529
  primary stream: completed lift 12/23; survivors=19,041,056
  primary stream: completed lift 18/23; survivors=28,561,586
  primary stream: completed lift 23/23; survivors=36,495,360
  independent chunked stream: completed lift 1/23; survivors=1,586,754
  independent chunked stream: completed lift 6/23; survivors=9,520,529
  independent chunked stream: completed lift 12/23; survivors=19,041,056
  independent chunked stream: completed lift 18/23; survivors=28,561,586
  independent chunked stream: completed lift 23/23; survivors=36,495,360
PN1C provenance
  protocol_sha256: 7DAA061BA790B12461ED60136FD9C50F3A36C10BED472819CFCC08B4B3462DBF
  target_gap_sha256: F68A8E707D9836C03C8FE5A84AD3FD3CA397F1736562CC2279094B631585923C
  target slots: 36,495,360
  exact checks: True
  primary pass: False
  split robustness: False
  rating: NOT SUPPORTED [pre-registered compression adva

## 2. Primary comparison

Lower Jensen–Shannon divergence means the compressed parent predicts the held-out child more closely. High-budget and exact-parent models are displayed as reference ceilings but are not eligible for the primary ≤36-slot contest.


In [2]:
scores = pd.read_csv(HERE / "PN1C_MODEL_SCORES.csv")
columns = ["model", "slots", "eligible_primary", "jsd_bits", "pair_jsd_mean_bits", "gain_over_uniform_per_slot"]
print(scores[columns].to_string(index=False, float_format=lambda value: f"{value:.6f}"))

eligible = scores[scores["eligible_primary"]].sort_values("jsd_bits")
ara = float(eligible.loc[eligible["model"] == "ARA-linear-6", "jsd_bits"].iloc[0])
winner = eligible.iloc[0]
print(f"\nEligible winner: {winner['model']} at {winner['jsd_bits']:.6f} bits")
print(f"ARA / winner divergence ratio: {ara / float(winner['jsd_bits']):.3f}x")


                model  slots  eligible_primary  jsd_bits  pair_jsd_mean_bits  gain_over_uniform_per_slot
Exact parent relation    575             False  0.003793            0.000820                    0.001044
           Gap-Markov    271             False  0.063456            0.000820                    0.001995
              Gap-IID     31              True  0.230999            0.059831                    0.012038
 Top-9 constellations     36              True  0.419932            0.125081                    0.005118
                DCT-6     36              True  0.428965            0.134530                    0.004867
          Log-ratio-6     35              True  0.454302            0.139446                    0.004283
         ARA-linear-6     35              True  0.470280            0.159464                    0.003826
   Learned-quantile-5     28              True  0.496182            0.177747                    0.003857
              Uniform      0             False  0.60419

![Frozen PN1C scores and budget frontier](PN1C_COMPRESSION_FIGURE.png)

The ARA grid improves on uniform, but its divergence is 2.036 times the winning gap-IID divergence. The 1% superiority threshold is therefore missed decisively rather than narrowly.


## 3. Robustness, target geometry and diagnosis

The target is a discrete web rather than a smooth field. Uniform decompression spreads each ARA coarse-cell mass over all 16 fine cells inside it. A gap marginal keeps the allowed gap identities and projects their combinations into the same ARA plane.


In [3]:
split = pd.read_csv(HERE / "PN1C_SPLIT_HALF.csv")
frontier = pd.read_csv(HERE / "PN1C_BUDGET_FRONTIER.csv")
checks = pd.read_csv(HERE / "PN1C_CALIBRATION_CHECKS.csv")
print("Split halves:")
print(split.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
print("\nFixed-coordinate budget frontier:")
print(frontier.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
print(f"\nExact checks passing: {int(checks['passes'].sum())}/{len(checks)}")

with np.load(HERE / "PN1C_TARGET_AND_PREDICTIONS.npz") as archive:
    target = archive["target_counts"].astype(float)
    target /= target.sum()
    ara_prediction = archive["prediction_ARA_linear_6"]
    gap_prediction = archive["prediction_Gap_IID"]
    support = target > 0
    print(f"Target-support mass — ARA: {ara_prediction[support].sum():.6f}")
    print(f"Target-support mass — Gap-IID: {gap_prediction[support].sum():.6f}")


Split halves:
 half  target_triples  ara_jsd_bits  gap_iid_jsd_bits  ara_beats_gap_iid
    1        18247680      0.470292          0.230997              False
    2        18247680      0.470271          0.231003              False

Fixed-coordinate budget frontier:
    family  retained_side  slots  jsd_bits
ARA-linear              4     15  0.503902
 Log-ratio              4     15  0.501625
       DCT              4     16  0.473900
ARA-linear              6     35  0.470280
 Log-ratio              6     35  0.454302
       DCT              6     36  0.428965
ARA-linear              8     63  0.417866
 Log-ratio              8     63  0.429167
       DCT              8     64  0.430439

Exact checks passing: 14/14
Target-support mass — ARA: 0.848011
Target-support mass — Gap-IID: 0.992802


![PN1C target and decompression diagnostic](PN1C_DISTRIBUTION_DIAGNOSTIC.png)

The gap model assigns 99.28% of its probability to cells that occur in the child; ARA assigns 84.80%. This locates the loss in the frozen coarse-graining/decompression rule, not in absence of a parent-to-child relation.


## 4. Independent reconstruction

The audit below does not import the primary analysis. It generates residues through repeated modulus filtering, materializes all 36,495,360 child gaps, counts circular triples through indexed chunks, and independently rebuilds every model and metric.


In [4]:
import pn1c_independent_validator as validator

validator.main()
with (HERE / "PN1C_INDEPENDENT_VALIDATION.json").open(encoding="utf-8") as handle:
    validation = json.load(handle)
print("\nIndependent validation summary:")
print(json.dumps({
    "all_checks_pass": validation["all_checks_pass"],
    "target_gap_sha256": validation["target_gap_sha256"],
    "max_prediction_abs_error": validation["max_prediction_abs_error"],
    "max_metric_abs_error": validation["max_metric_abs_error"],
    "independent_primary_winner": validation["independent_primary_winner"],
}, indent=2))


{
  "test_id": "T228 / PN1C/v1 independent validation",
  "method": "Standalone repeated-modulus residue filtering; materialized child gap cycle; indexed circular chunk counts; standalone model and metric reconstruction.",
  "protocol_sha256": "7DAA061BA790B12461ED60136FD9C50F3A36C10BED472819CFCC08B4B3462DBF",
  "target_gap_sha256": "F68A8E707D9836C03C8FE5A84AD3FD3CA397F1736562CC2279094B631585923C",
  "checks": {
    "protocol_hash_matches": true,
    "parent_period_exact": true,
    "parent_residue_count_exact": true,
    "parent_gap_sum_exact": true,
    "child_gap_count_exact": true,
    "child_gap_sum_exact": true,
    "child_gaps_positive_even": true,
    "target_counts_equal_archive": true,
    "half_counts_equal_archive": true,
    "parent_counts_equal_archive": true,
    "all_predictions_reconstructed": true,
    "all_metrics_recomputed": true,
    "gap_hash_matches_results": true,
    "primary_winner_recovered": true,
    "primary_failure_recovered": true
  },
  "all_checks_pa

## 5. Conclusion and next clean test

PN1C rejects the fixed-grid compression advantage. The result suggests that a future ARA state needs a predeclared support-, identity- or transition-aware decompression law if it is to preserve this arithmetic web under strong compression.

Prime 23 is now development data. Any revised model must be chosen using rungs only through 23, frozen, and then tested on an unopened prime-29 wheel. A stronger comparison should use literal fixed-bit or minimum-description-length budgets because scalar-slot counts do not charge all labels, probabilities and fixed algorithms by their true encoding cost.

The post-open repair was serialization-only: pandas changed the Uniform reference's undefined gain-per-slot from `None` to `NaN`, which strict JSON rejected. The writer now maps that display value to JSON `null`; no mathematical object or result changed.


## Provenance

- Test ID: `T228 / PN1C/v1`
- Protocol: `PN1C_COMPRESSION_PROTOCOL_v1_FROZEN.md`
- Primary analysis: `pn1c_compression_test.py`
- Independent audit: `pn1c_independent_validator.py`
- Canonical result: `PN1C_COMPRESSION_RESULTS.json`
- Full written interpretation: `PN1C_COMPRESSION_RESULT.md`
